# Deep Learning — Module 2: Training & Optimization · Part 5
## Full Training Loops & Module 2 Master Q&A

> This notebook ties everything from Module 2 together into complete, runnable training loops.

---

## Table of Contents

| Section | Topic |
|---------|-------|
| **1** | Full NumPy Training Loop from Scratch |
| **1.1** | Forward Pass (all layers) |
| **1.2** | Loss + Gradient Computation |
| **1.3** | Optimizer Step (SGD + Momentum) |
| **1.4** | Validation Loop + Early Stopping |
| **1.5** | Training Curves + Decision Boundary |
| **2** | PyTorch Production-Grade Training Loop |
| **2.1** | Model + DataLoader Setup |
| **2.2** | Optimizer + Scheduler |
| **2.3** | Mixed Precision Training (AMP) |
| **2.4** | Gradient Clipping |
| **2.5** | Checkpointing + Resume |
| **2.6** | Full Loop with All Best Practices |
| **3** | Module 2 Master Interview Q&A Cheatsheet |


In [ ]:
# ── All imports ─────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 12,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 120,
    "axes.prop_cycle": plt.cycler(color=[
        "#4C72B0","#DD8452","#55A868","#C44E52","#8172B3","#937860"
    ])
})
print("Imports ready ✓")


---

## 1. Full NumPy Training Loop from Scratch

Building a **complete, production-style** training loop using only NumPy:
- 3-layer MLP (2 hidden layers + output)
- BatchNorm (manual)
- Dropout (inverted)
- SGD with Momentum
- Early stopping
- Validation at every epoch

This encodes everything from Module 2 in one unified script.

### Network Architecture
```
Input (n_in)
  ↓
Linear → BatchNorm → ReLU → Dropout
  ↓
Linear → BatchNorm → ReLU → Dropout
  ↓
Linear → Sigmoid (binary classification)
  ↓
BCE Loss
```


In [ ]:
import numpy as np

np.random.seed(42)

# ══════════════════════════════════════════════════════════════════
#  UTILITY FUNCTIONS
# ══════════════════════════════════════════════════════════════════

def sigmoid(z):         return 1 / (1 + np.exp(-np.clip(z, -500, 500)))
def sigmoid_d(z):       s = sigmoid(z); return s * (1 - s)
def relu(z):            return np.maximum(0, z)
def relu_d(z):          return (z > 0).astype(float)
def bce(yhat, y, eps=1e-9):
    return -np.mean(y * np.log(yhat + eps) + (1 - y) * np.log(1 - yhat + eps))

# ══════════════════════════════════════════════════════════════════
#  BATCH NORMALIZATION (manual)
# ══════════════════════════════════════════════════════════════════
class BatchNorm:
    def __init__(self, n, momentum=0.1, eps=1e-5):
        self.gamma   = np.ones(n)
        self.beta    = np.zeros(n)
        self.momentum = momentum
        self.eps      = eps
        self.running_mean = np.zeros(n)
        self.running_var  = np.ones(n)
        # cache for backprop
        self.cache = {}

    def forward(self, z, training=True):
        if training:
            mu  = z.mean(axis=0)
            var = z.var(axis=0)
            self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * mu
            self.running_var  = (1 - self.momentum) * self.running_var  + self.momentum * var
        else:
            mu  = self.running_mean
            var = self.running_var
        z_hat = (z - mu) / np.sqrt(var + self.eps)
        out   = self.gamma * z_hat + self.beta
        self.cache = {'z': z, 'mu': mu, 'var': var, 'z_hat': z_hat, 'training': training}
        return out

    def backward(self, d_out):
        z, mu, var, z_hat = self.cache['z'], self.cache['mu'], self.cache['var'], self.cache['z_hat']
        B = z.shape[0]
        std_inv = 1.0 / np.sqrt(var + self.eps)
        d_z_hat  = d_out * self.gamma
        d_var    = (-0.5 * d_z_hat * (z - mu) * std_inv**3).sum(axis=0)
        d_mu     = (-d_z_hat * std_inv).sum(axis=0) + d_var * (-2 * (z - mu)).mean(axis=0)
        d_z      = d_z_hat * std_inv + d_var * 2*(z - mu)/B + d_mu / B
        d_gamma  = (d_out * z_hat).sum(axis=0)
        d_beta   = d_out.sum(axis=0)
        return d_z, d_gamma, d_beta

# ══════════════════════════════════════════════════════════════════
#  INVERTED DROPOUT
# ══════════════════════════════════════════════════════════════════
class Dropout:
    def __init__(self, p=0.3):
        self.p = p
        self.mask = None
    def forward(self, x, training=True):
        if not training or self.p == 0:
            return x
        self.mask = (np.random.rand(*x.shape) > self.p) / (1 - self.p)
        return x * self.mask
    def backward(self, d_out):
        return d_out * self.mask if self.mask is not None else d_out

print("BatchNorm and Dropout classes defined ✓")


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  3-LAYER MLP with BN + Dropout
# ══════════════════════════════════════════════════════════════════
class MLP:
    def __init__(self, n_in, n_h1, n_h2, n_out, drop_p=0.3):
        # He init for ReLU layers
        scale1 = np.sqrt(2.0 / n_in)
        scale2 = np.sqrt(2.0 / n_h1)
        self.W1 = np.random.randn(n_in, n_h1) * scale1
        self.b1 = np.zeros(n_h1)
        self.W2 = np.random.randn(n_h1, n_h2) * scale2
        self.b2 = np.zeros(n_h2)
        self.W3 = np.random.randn(n_h2, n_out) * np.sqrt(2.0 / n_h2)
        self.b3 = np.zeros(n_out)

        self.bn1 = BatchNorm(n_h1)
        self.bn2 = BatchNorm(n_h2)
        self.drop1 = Dropout(drop_p)
        self.drop2 = Dropout(drop_p)
        self.cache = {}

    def forward(self, X, training=True):
        # Layer 1
        Z1 = X @ self.W1 + self.b1
        N1 = self.bn1.forward(Z1, training)
        A1 = relu(N1)
        D1 = self.drop1.forward(A1, training)
        # Layer 2
        Z2 = D1 @ self.W2 + self.b2
        N2 = self.bn2.forward(Z2, training)
        A2 = relu(N2)
        D2 = self.drop2.forward(A2, training)
        # Output
        Z3 = D2 @ self.W3 + self.b3
        A3 = sigmoid(Z3)

        self.cache = {'X': X, 'Z1': Z1, 'N1': N1, 'A1': A1, 'D1': D1,
                      'Z2': Z2, 'N2': N2, 'A2': A2, 'D2': D2, 'Z3': Z3}
        return A3

    def backward(self, A3, y):
        B = y.shape[0]
        # ── Output layer ──
        dZ3 = (A3 - y.reshape(-1, 1)) / B        # BCE + sigmoid → ŷ - y
        dW3 = self.cache['D2'].T @ dZ3
        db3 = dZ3.sum(axis=0)
        dD2 = dZ3 @ self.W3.T
        # ── Dropout 2 ──
        dA2 = self.drop2.backward(dD2)
        # ── Layer 2 ──
        dN2 = dA2 * relu_d(self.cache['N2'])
        dZ2, dg2, dbt2 = self.bn2.backward(dN2)
        dW2 = self.cache['D1'].T @ dZ2
        db2 = dZ2.sum(axis=0)
        dD1 = dZ2 @ self.W2.T
        # ── Dropout 1 ──
        dA1 = self.drop1.backward(dD1)
        # ── Layer 1 ──
        dN1 = dA1 * relu_d(self.cache['N1'])
        dZ1, dg1, dbt1 = self.bn1.backward(dN1)
        dW1 = self.cache['X'].T @ dZ1
        db1 = dZ1.sum(axis=0)

        return {'W1': dW1, 'b1': db1, 'W2': dW2, 'b2': db2,
                'W3': dW3, 'b3': db3,
                'bn1_gamma': dg1, 'bn1_beta': dbt1,
                'bn2_gamma': dg2, 'bn2_beta': dbt2}

    def predict(self, X):
        return self.forward(X, training=False)

print("MLP class defined ✓")


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  SGD WITH MOMENTUM OPTIMIZER
# ══════════════════════════════════════════════════════════════════
class SGDMomentum:
    def __init__(self, model, lr=0.05, momentum=0.9, weight_decay=1e-4):
        self.model        = model
        self.lr           = lr
        self.momentum     = momentum
        self.weight_decay = weight_decay
        # Velocity dict, same keys as grads
        self.v = {k: np.zeros_like(getattr(model, k, np.array([])))
                  for k in ['W1','b1','W2','b2','W3','b3']}
        self.v['bn1_gamma'] = np.zeros_like(model.bn1.gamma)
        self.v['bn1_beta']  = np.zeros_like(model.bn1.beta)
        self.v['bn2_gamma'] = np.zeros_like(model.bn2.gamma)
        self.v['bn2_beta']  = np.zeros_like(model.bn2.beta)

    def step(self, grads):
        for key, grad in grads.items():
            param = None
            if   key == 'W1': param = self.model.W1
            elif key == 'b1': param = self.model.b1
            elif key == 'W2': param = self.model.W2
            elif key == 'b2': param = self.model.b2
            elif key == 'W3': param = self.model.W3
            elif key == 'b3': param = self.model.b3
            elif key == 'bn1_gamma': param = self.model.bn1.gamma
            elif key == 'bn1_beta':  param = self.model.bn1.beta
            elif key == 'bn2_gamma': param = self.model.bn2.gamma
            elif key == 'bn2_beta':  param = self.model.bn2.beta

            if param is None: continue
            g = grad + (self.weight_decay * param if 'W' in key else 0)
            self.v[key] = self.momentum * self.v[key] + g
            param -= self.lr * self.v[key]

print("SGDMomentum defined ✓")


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  DATASET + FULL TRAINING LOOP
# ══════════════════════════════════════════════════════════════════
np.random.seed(42)

# ── Concentric circles dataset ───────────────────────────────────
def make_circles(N=600, noise=0.12):
    n = N // 2
    theta = np.linspace(0, 2*np.pi, n)
    r_in  = 0.5 + noise * np.random.randn(n)
    r_out = 1.2 + noise * np.random.randn(n)
    X0 = np.c_[r_in  * np.cos(theta), r_in  * np.sin(theta)]
    X1 = np.c_[r_out * np.cos(theta), r_out * np.sin(theta)]
    X  = np.vstack([X0, X1])
    y  = np.hstack([np.zeros(n), np.ones(n)])
    perm = np.random.permutation(N)
    return X[perm], y[perm]

X_all, y_all = make_circles(600)
split = int(0.8 * len(X_all))
X_tr, y_tr   = X_all[:split], y_all[:split]
X_val, y_val = X_all[split:], y_all[split:]

print(f"Train: {X_tr.shape[0]} | Val: {X_val.shape[0]}")

# ── Model + Optimizer ────────────────────────────────────────────
model = MLP(n_in=2, n_h1=64, n_h2=64, n_out=1, drop_p=0.3)
optimizer = SGDMomentum(model, lr=0.05, momentum=0.9, weight_decay=1e-4)

# ── Training parameters ───────────────────────────────────────────
EPOCHS     = 300
BATCH_SIZE = 64
PATIENCE   = 20

# ── Training history ─────────────────────────────────────────────
tr_losses, val_losses, tr_accs, val_accs = [], [], [], []
best_val_loss = np.inf
patience_count = 0
best_weights = None

def get_weights(m):
    return {k: v.copy() for k, v in {
        'W1':m.W1,'b1':m.b1,'W2':m.W2,'b2':m.b2,'W3':m.W3,'b3':m.b3,
        'bn1_g':m.bn1.gamma,'bn1_b':m.bn1.beta,
        'bn2_g':m.bn2.gamma,'bn2_b':m.bn2.beta,
        'bn1_rm':m.bn1.running_mean,'bn1_rv':m.bn1.running_var,
        'bn2_rm':m.bn2.running_mean,'bn2_rv':m.bn2.running_var,
    }.items()}

def set_weights(m, w):
    m.W1[:]=w['W1'];m.b1[:]=w['b1'];m.W2[:]=w['W2'];m.b2[:]=w['b2']
    m.W3[:]=w['W3'];m.b3[:]=w['b3']
    m.bn1.gamma[:]=w['bn1_g'];m.bn1.beta[:]=w['bn1_b']
    m.bn2.gamma[:]=w['bn2_g'];m.bn2.beta[:]=w['bn2_b']
    m.bn1.running_mean[:]=w['bn1_rm'];m.bn1.running_var[:]=w['bn1_rv']
    m.bn2.running_mean[:]=w['bn2_rm'];m.bn2.running_var[:]=w['bn2_rv']

# ════════════════════════════
#  MAIN TRAINING LOOP
# ════════════════════════════
for epoch in range(EPOCHS):

    # ── Shuffle ──
    perm = np.random.permutation(len(X_tr))
    X_sh, y_sh = X_tr[perm], y_tr[perm]

    # ── Mini-batch loop ──
    epoch_loss = 0.0
    n_batches  = 0
    for start in range(0, len(X_tr), BATCH_SIZE):
        end = start + BATCH_SIZE
        Xb, yb = X_sh[start:end], y_sh[start:end]

        # Forward
        yhat = model.forward(Xb, training=True)
        loss = bce(yhat, yb)
        epoch_loss += loss

        # Backward
        grads = model.backward(yhat, yb)

        # Gradient clipping (norm)
        all_grads = np.concatenate([g.flatten() for g in grads.values()])
        grad_norm = np.linalg.norm(all_grads)
        clip_val  = 1.0
        if grad_norm > clip_val:
            scale = clip_val / grad_norm
            grads = {k: v * scale for k, v in grads.items()}

        # Optimizer step
        optimizer.step(grads)
        n_batches += 1

    # ── Validation ──
    tr_yhat  = model.predict(X_tr)
    val_yhat = model.predict(X_val)
    tr_l     = bce(tr_yhat, y_tr)
    val_l    = bce(val_yhat, y_val)
    tr_acc   = ((tr_yhat.flatten() > 0.5) == y_tr).mean()
    val_acc  = ((val_yhat.flatten() > 0.5) == y_val).mean()

    tr_losses.append(tr_l); val_losses.append(val_l)
    tr_accs.append(tr_acc); val_accs.append(val_acc)

    # ── Early stopping ──
    if val_l < best_val_loss - 1e-5:
        best_val_loss = val_l
        patience_count = 0
        best_weights = get_weights(model)
    else:
        patience_count += 1
    if patience_count >= PATIENCE:
        print(f"Early stopping at epoch {epoch+1}")
        break

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1:3d} | train_loss={tr_l:.4f} acc={tr_acc*100:.1f}% | "
              f"val_loss={val_l:.4f} acc={val_acc*100:.1f}%")

# Restore best weights
set_weights(model, best_weights)
final_val_yhat = model.predict(X_val)
print(f"\nFinal Val Loss: {bce(final_val_yhat, y_val):.4f} | "
      f"Acc: {((final_val_yhat.flatten()>0.5)==y_val).mean()*100:.1f}%")


In [ ]:
# Figure 1: Full NumPy Training Loop — curves + decision boundary
fig = plt.figure(figsize=(16, 5))
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.3)

# ── Loss curves ──
ax0 = fig.add_subplot(gs[0])
ep = range(1, len(tr_losses)+1)
ax0.plot(ep, tr_losses,  lw=2, color='#4C72B0', label='Train Loss')
ax0.plot(ep, val_losses, lw=2, color='#C44E52', label='Val Loss')
ax0.set_xlabel('Epoch'); ax0.set_ylabel('BCE Loss')
ax0.set_title('Training & Validation Loss'); ax0.legend()

# ── Accuracy curves ──
ax1 = fig.add_subplot(gs[1])
ax1.plot(ep, [a*100 for a in tr_accs],  lw=2, color='#55A868', label='Train Acc')
ax1.plot(ep, [a*100 for a in val_accs], lw=2, color='#DD8452', label='Val Acc')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy (%)')
ax1.set_title('Training & Validation Accuracy'); ax1.legend()

# ── Decision boundary ──
ax2 = fig.add_subplot(gs[2])
xx, yy = np.meshgrid(np.linspace(-1.8, 1.8, 150), np.linspace(-1.8, 1.8, 150))
grid   = np.c_[xx.ravel(), yy.ravel()]
probs  = model.predict(grid).reshape(xx.shape)
ax2.contourf(xx, yy, probs, levels=20, cmap='RdBu', alpha=0.75, vmin=0, vmax=1)
ax2.contour( xx, yy, probs, levels=[0.5], colors='white', linewidths=2.5)
ax2.scatter(X_val[y_val==0,0], X_val[y_val==0,1], c='#4C72B0', s=25, alpha=0.7, label='Class 0')
ax2.scatter(X_val[y_val==1,0], X_val[y_val==1,1], c='#C44E52', s=25, alpha=0.7, label='Class 1')
ax2.set_title('Learned Decision Boundary (NumPy MLP)'); ax2.legend(fontsize=9)

plt.suptitle('Figure 1: NumPy Full Training Loop (BN + Dropout + Momentum + EarlyStop)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("\nImplemented in NumPy:")
print("  ✓ He initialization")
print("  ✓ Batch Normalization (forward + backward + running stats)")
print("  ✓ Inverted Dropout")
print("  ✓ SGD with Momentum + Weight Decay")
print("  ✓ Gradient Clipping (by norm)")
print("  ✓ Early Stopping + Best Checkpoint Restore")


---

## 2. PyTorch Production-Grade Training Loop

The same problem, now using PyTorch with **all best practices you'd find in a real ML codebase**:

| Feature | NumPy (Section 1) | PyTorch (Section 2) |
|---------|-----------------|-------------------|
| Autograd | ❌ Manual backprop | ✅ Automatic |
| BatchNorm | ✅ Manual | ✅ `nn.BatchNorm1d` |
| Dropout | ✅ Manual inverted | ✅ `nn.Dropout` |
| Optimizer | ✅ Manual SGD+Mom | ✅ `torch.optim.AdamW` |
| LR Scheduling | ❌ Fixed | ✅ `CosineAnnealingLR` |
| Gradient clipping | ✅ Manual | ✅ `clip_grad_norm_` |
| Mixed precision | ❌ | ✅ `torch.cuda.amp` |
| Checkpointing | ✅ Manual | ✅ `torch.save` / `load` |
| DataLoader | ❌ Manual batching | ✅ `DataLoader` |
| Early stopping | ✅ Manual | ✅ Custom class |


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import os

torch.manual_seed(42)
np.random.seed(42)

# ══════════════════════════════════════════════════════════════════
#  DATASET — same concentric circles as NumPy section
# ══════════════════════════════════════════════════════════════════
def make_circles_torch(N=600, noise=0.12, seed=42):
    np.random.seed(seed)
    n = N // 2
    theta = np.linspace(0, 2*np.pi, n)
    X0 = np.c_[0.5*np.cos(theta) + noise*np.random.randn(n),
               0.5*np.sin(theta) + noise*np.random.randn(n)]
    X1 = np.c_[1.2*np.cos(theta) + noise*np.random.randn(n),
               1.2*np.sin(theta) + noise*np.random.randn(n)]
    X  = np.vstack([X0, X1]).astype(np.float32)
    y  = np.hstack([np.zeros(n), np.ones(n)]).astype(np.float32)
    perm = np.random.permutation(N)
    return torch.from_numpy(X[perm]), torch.from_numpy(y[perm])

X_all, y_all = make_circles_torch()
split = int(0.8 * len(X_all))
X_tr, y_tr   = X_all[:split], y_all[:split].unsqueeze(1)
X_val, y_val = X_all[split:], y_all[split:].unsqueeze(1)

# DataLoaders
train_ds  = TensorDataset(X_tr, y_tr)
val_ds    = TensorDataset(X_val, y_val)
train_dl  = DataLoader(train_ds, batch_size=64, shuffle=True,  drop_last=True)
val_dl    = DataLoader(val_ds,   batch_size=128, shuffle=False)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")
print(f"Train batches: {len(train_dl)} | Val batches: {len(val_dl)}")

# ══════════════════════════════════════════════════════════════════
#  MODEL — MLP with BatchNorm + Dropout
# ══════════════════════════════════════════════════════════════════
class DeepMLP(nn.Module):
    def __init__(self, n_in=2, n_h=64, n_layers=3, dropout_p=0.3):
        super().__init__()
        layers = []
        in_dim = n_in
        for _ in range(n_layers):
            layers += [
                nn.Linear(in_dim, n_h),
                nn.BatchNorm1d(n_h),
                nn.ReLU(),
                nn.Dropout(dropout_p),
            ]
            in_dim = n_h
        layers.append(nn.Linear(n_h, 1))  # output (raw logit)
        self.net = nn.Sequential(*layers)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.ones_(m.weight)   # γ = 1
                nn.init.zeros_(m.bias)    # β = 0

    def forward(self, x):
        return self.net(x)

model = DeepMLP()
total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nModel parameters: {total_params:,} total, {trainable:,} trainable")
print(model)


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  EARLY STOPPING + CHECKPOINTING
# ══════════════════════════════════════════════════════════════════
class EarlyStopping:
    def __init__(self, patience=15, min_delta=1e-5, checkpoint_path='best_model.pt'):
        self.patience   = patience
        self.min_delta  = min_delta
        self.ckpt_path  = checkpoint_path
        self.counter    = 0
        self.best_loss  = float('inf')
        self.stop       = False

    def __call__(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter   = 0
            torch.save({
                'model_state':     model.state_dict(),
                'val_loss':        val_loss,
            }, self.ckpt_path)
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True

    def load_best(self, model):
        ckpt = torch.load(self.ckpt_path, weights_only=True)
        model.load_state_dict(ckpt['model_state'])
        print(f"Restored best model (val_loss={ckpt['val_loss']:.4f})")
        return model

print("EarlyStopping class defined ✓")


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  OPTIMIZER + LR SCHEDULER + AMP SCALER
# ══════════════════════════════════════════════════════════════════
EPOCHS     = 250
LR         = 1e-3
WEIGHT_DEC = 0.01
T_WARM     = 10     # warmup epochs
MAX_NORM   = 1.0    # gradient clip norm

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DEC)

# LR Schedule: Linear warmup → Cosine decay
warmup_scheduler = optim.lr_scheduler.LinearLR(
    optimizer, start_factor=0.01, end_factor=1.0, total_iters=T_WARM)
cosine_scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS - T_WARM, eta_min=1e-6)
scheduler = optim.lr_scheduler.SequentialLR(
    optimizer, [warmup_scheduler, cosine_scheduler], milestones=[T_WARM])

# AMP (Automatic Mixed Precision) — uses float16 for speed on CUDA
# On CPU this is a no-op (float32 only)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
use_amp = device == 'cuda'
scaler  = torch.cuda.amp.GradScaler(enabled=use_amp)
model   = model.to(device)
print(f"Device: {device} | AMP: {use_amp}")
print(f"Optimizer: AdamW (lr={LR}, wd={WEIGHT_DEC})")
print(f"Scheduler: Warmup ({T_WARM} ep) + CosineAnnealing")
print(f"Grad clip: max_norm={MAX_NORM}")


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  PRODUCTION TRAINING LOOP
# ══════════════════════════════════════════════════════════════════
criterion    = nn.BCEWithLogitsLoss()
early_stop   = EarlyStopping(patience=20, checkpoint_path='best_model.pt')

history = {'tr_loss': [], 'val_loss': [], 'tr_acc': [], 'val_acc': [], 'lr': [], 'grad_norm': []}

for epoch in range(EPOCHS):

    # ════════════════════════════
    #  TRAINING PHASE
    # ════════════════════════════
    model.train()
    batch_losses, batch_accs, batch_gnorms = [], [], []

    for Xb, yb in train_dl:
        Xb, yb = Xb.to(device), yb.to(device)
        optimizer.zero_grad()

        # ① Forward (with AMP context)
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
            logits = model(Xb)
            loss   = criterion(logits, yb)

        # ② Backward (AMP-scaled)
        scaler.scale(loss).backward()

        # ③ Unscale gradients before clipping
        scaler.unscale_(optimizer)
        grad_norm = nn.utils.clip_grad_norm_(model.parameters(), MAX_NORM)

        # ④ Optimizer step (AMP-aware)
        scaler.step(optimizer)
        scaler.update()

        preds = (logits.detach().sigmoid() > 0.5).float()
        batch_losses.append(loss.item())
        batch_accs.append((preds == yb).float().mean().item())
        batch_gnorms.append(grad_norm.item() if torch.is_tensor(grad_norm) else grad_norm)

    # ④ LR Scheduler step (once per epoch)
    scheduler.step()

    # ════════════════════════════
    #  VALIDATION PHASE
    # ════════════════════════════
    model.eval()
    val_losses_ep, val_accs_ep = [], []
    with torch.no_grad():
        for Xb, yb in val_dl:
            Xb, yb = Xb.to(device), yb.to(device)
            logits  = model(Xb)
            v_loss  = criterion(logits, yb)
            v_preds = (logits.sigmoid() > 0.5).float()
            val_losses_ep.append(v_loss.item())
            val_accs_ep.append((v_preds == yb).float().mean().item())

    tr_l  = np.mean(batch_losses)
    val_l = np.mean(val_losses_ep)
    tr_a  = np.mean(batch_accs)
    val_a = np.mean(val_accs_ep)
    cur_lr = optimizer.param_groups[0]['lr']
    gn     = np.mean(batch_gnorms)

    history['tr_loss'].append(tr_l); history['val_loss'].append(val_l)
    history['tr_acc'].append(tr_a);  history['val_acc'].append(val_a)
    history['lr'].append(cur_lr);    history['grad_norm'].append(gn)

    # Early stopping check
    early_stop(val_l, model)
    if early_stop.stop:
        print(f"Early stopping at epoch {epoch+1}")
        break

    if (epoch+1) % 50 == 0:
        print(f"Epoch {epoch+1:3d} | lr={cur_lr:.2e} | "
              f"train: loss={tr_l:.4f} acc={tr_a*100:.1f}% | "
              f"val: loss={val_l:.4f} acc={val_a*100:.1f}% | gnorm={gn:.3f}")

# Restore best weights
model = early_stop.load_best(model)


In [ ]:
# Figure 2: PyTorch Production Training Loop — 4-panel dashboard
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
ep = range(1, len(history['tr_loss'])+1)

# ── Loss ──
axes[0,0].plot(ep, history['tr_loss'],  lw=2, color='#4C72B0', label='Train')
axes[0,0].plot(ep, history['val_loss'], lw=2, color='#C44E52', label='Val')
axes[0,0].set_xlabel('Epoch'); axes[0,0].set_ylabel('BCE Loss')
axes[0,0].set_title('Loss Curves'); axes[0,0].legend()

# ── Accuracy ──
axes[0,1].plot(ep, [a*100 for a in history['tr_acc']],  lw=2, color='#55A868', label='Train')
axes[0,1].plot(ep, [a*100 for a in history['val_acc']], lw=2, color='#DD8452', label='Val')
axes[0,1].set_xlabel('Epoch'); axes[0,1].set_ylabel('Accuracy (%)')
axes[0,1].set_title('Accuracy Curves'); axes[0,1].legend()

# ── Learning Rate ──
axes[1,0].plot(ep, history['lr'], lw=2, color='#8172B3')
axes[1,0].axvspan(1, min(10, len(ep)), alpha=0.12, color='gold', label='Warmup')
axes[1,0].set_xlabel('Epoch'); axes[1,0].set_ylabel('Learning Rate')
axes[1,0].set_title('LR Schedule: Warmup + Cosine Decay'); axes[1,0].legend()

# ── Gradient Norm ──
axes[1,1].plot(ep, history['grad_norm'], lw=1.5, color='#937860', alpha=0.7, label='Grad norm')
axes[1,1].axhline(1.0, color='#C44E52', ls='--', lw=1.5, label='Clip threshold=1.0')
axes[1,1].set_xlabel('Epoch'); axes[1,1].set_ylabel('||∇L||')
axes[1,1].set_title('Gradient Norm (with clipping)'); axes[1,1].legend()

plt.suptitle('Figure 2: PyTorch Production Training Loop Dashboard',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Final evaluation
model.eval()
with torch.no_grad():
    val_preds = (model(X_val.to(device)).sigmoid().cpu() > 0.5).float()
final_acc = (val_preds == y_val).float().mean().item()
print(f"\nFinal Val Accuracy: {final_acc*100:.2f}%")


In [ ]:
# ── Training loop 5-step checklist ──────────────────────────────
print("═══ PyTorch Training Loop Checklist ══════════════════════════")
print()
print("Per epoch:")
print("  model.train()                              # activate BN + Dropout")
print()
print("Per batch:")
print("  1. optimizer.zero_grad()                   # clear old gradients")
print("  2. logits = model(Xb)                      # forward pass")
print("  3. loss = criterion(logits, yb)            # compute loss")
print("  4. scaler.scale(loss).backward()           # backprop (AMP-scaled)")
print("  5. scaler.unscale_(optimizer)              # unscale before clipping")
print("  6. clip_grad_norm_(model.parameters(), 1.0) # gradient clipping")
print("  7. scaler.step(optimizer)                  # weight update (AMP-aware)")
print("  8. scaler.update()                         # update AMP scale factor")
print()
print("Per epoch (after batches):")
print("  scheduler.step()                           # LR schedule update")
print()
print("Validation (every N epochs):")
print("  model.eval()                               # disable BN + Dropout")
print("  with torch.no_grad():                      # no grad tracking")
print("      ...                                    # forward + metrics")
print()
print("Early stopping:")
print("  early_stop(val_loss, model)                # saves best checkpoint")
print("  early_stop.load_best(model)                # restore best weights")


---

## 3. Module 2 — Master Interview Q&A Cheatsheet

This is the **complete reference** covering all of Module 2:
Loss Functions · Backpropagation · Optimizers · LR Scheduling · Regularization · Normalization · Initialization

---

### Level 1 — Beginner (Entry Level / Internship)

> **Q: What are the 4 steps of neural network training?**
> A: (1) **Forward pass** — compute predictions. (2) **Loss computation** — measure error. (3) **Backpropagation** — compute gradients via chain rule. (4) **Optimizer step** — update weights in gradient direction.

> **Q: What is the difference between MSE and Cross-Entropy loss?**
> A: MSE for regression (continuous outputs), Cross-Entropy for classification (probability outputs). MSE + sigmoid has vanishing gradients when confident and wrong; BCE + sigmoid has gradient = ŷ-y, which is large and clean in that case.

> **Q: What does learning rate control?**
> A: The size of each parameter update step. Too large → training diverges (overshoots minimum). Too small → training is slow and may get stuck. Most critical hyperparameter to tune.

> **Q: What is overfitting and name 3 ways to prevent it.**
> A: Overfitting = model memorises training data, fails on new data (low train loss, high val loss). Fixes: (1) L2 regularization / weight decay. (2) Dropout. (3) Early stopping. Also: more data, data augmentation, reduce model size.

> **Q: What does BatchNorm do?**
> A: Normalises each layer's inputs to zero mean and unit variance across the mini-batch, then applies learnable scale γ and shift β. Stabilises training, allows larger learning rates, reduces sensitivity to initialization.

> **Q: What initialization should you use for ReLU networks?**
> A: He (Kaiming) initialization — $\mathcal{N}(0, \sqrt{2/n_{in}})$. The factor of 2 compensates for ReLU zeroing half the pre-activations, preserving signal variance across depth.

---

### Level 2 — Mid-Level (1–3 years experience)

> **Q: Explain Adam optimizer — what are the two moments and how are they used?**
> A: **1st moment** $m_t$ = EMA of gradients → direction/momentum. **2nd moment** $v_t$ = EMA of squared gradients → per-parameter scale. Update: $W -= \eta \cdot \hat{m}_t / (\sqrt{\hat{v}_t}+\epsilon)$. Bias correction $\hat{m}_t = m_t/(1-\beta_1^t)$ fixes the cold-start initialisation at 0.

> **Q: Why does BatchNorm behave differently at train vs test time?**
> A: Training: uses current mini-batch statistics (noisy). Testing: uses running statistics accumulated over training (stable population estimates). `model.eval()` switches BN to use running stats. Forgetting it → inconsistent, often worse predictions.

> **Q: L1 vs L2 regularization — which produces sparse solutions and why?**
> A: L1. Geometrically: L1 constraint region is a diamond — loss contours meet it at corners (on coordinate axes), setting some weights to exactly 0. L2's circle has no corners — loss contours meet smoothly, shrinking but not zeroing weights.

> **Q: What is vanishing gradient and what causes it in deep networks?**
> A: Each layer's gradient is the product of local Jacobians along the chain rule. If each Jacobian has spectral norm < 1 (e.g., sigmoid derivative ≤ 0.25), the product decreases exponentially. Early layers receive near-zero gradients → don't learn. Fixed by: ReLU activations, BatchNorm, residual connections.

> **Q: Explain inverted dropout and why it's preferred over vanilla dropout.**
> A: Inverted dropout divides surviving activations by $(1-p)$ during training → expected output preserved. Vanilla dropout multiplies by $(1-p)$ at test time → extra inference compute and easy to forget. PyTorch uses inverted dropout — `nn.Dropout` does nothing at eval time.

> **Q: Why do transformers use LayerNorm instead of BatchNorm?**
> A: LayerNorm normalises per-sample across features → works at batch=1 (needed for auto-regressive generation), handles variable-length sequences, identical at train and test (no mode switching). BatchNorm normalises per-feature across the batch → fails at batch=1 and needs large batches.

> **Q: Derive why He init uses factor 2 vs Xavier.**
> A: Xavier: $\sigma^2_W = 2/(n_{in}+n_{out})$ for linear layers with identity activation. ReLU zeroes negative half → variance halved per layer. To compensate: $\sigma^2_W = 2/n_{in}$ (one-sided). He doubles the Xavier variance, preserved signal variance through ReLU depth.

---

### Level 3 — Senior MLE / Staff Engineer

> **Q: A 50-layer network shows NaN loss at step 1. What is your debugging process?**
> A: (1) Check init — zero init or very large random? (2) Check input data — NaN, inf, or extreme values? (3) Check loss function — `log(0)` in BCE, incorrect label range? (4) Try gradient clipping first as immediate fix. (5) Add BN layers immediately after linear layers. (6) Reduce LR by 10x. (7) Check for in-place ops on `requires_grad=True` tensors. (8) Use `torch.autograd.detect_anomaly()` to pinpoint the exact op that produces NaN.

> **Q: Compare SGD+Momentum vs Adam for production CV model training.**
> A: SGD+Momentum with cosine LR often achieves 0.5–1% better top-1 accuracy on ImageNet than Adam at the cost of harder tuning (LR very sensitive). Adam converges faster, is robust to LR choice, great for prototyping. Standard practice: use Adam early in research, switch to SGD+momentum with tuned LR for final production model. AdamW is a good middle ground — faster than SGD, better generalisation than Adam.

> **Q: Design a complete training setup for fine-tuning a ViT-Large on a 50k image dataset.**
> A: (1) **Optimizer**: AdamW, lr=1e-4 for backbone, 1e-3 for new head (different param groups). (2) **Scheduler**: linear warmup 5 epochs → cosine decay to 1e-6. (3) **Weight decay**: 0.05 (strong regularization for small dataset). (4) **Label smoothing**: 0.1. (5) **Dropout**: 0.1 (ViT already has it). (6) **Gradient clipping**: max_norm=1.0. (7) **BatchNorm**: LayerNorm already used in ViT. (8) **Data augmentation**: RandAugment + CutMix + MixUp. (9) **Early stopping**: patience=10 on val accuracy. (10) **Batch size**: 64 with gradient accumulation if GPU limited. (11) **Mixed precision**: AMP float16 throughout.

> **Q: Explain the interaction between BatchNorm and Dropout and why they're often not used together.**
> A: BatchNorm normalises using batch statistics that include Dropout's random zeroing noise. Running mean/var during training include this noise. At test time, Dropout is off but BatchNorm uses running stats computed with it — creating a "variance shift". Specifically, the variance of activations at test time differs from what BN expects. This degrades performance. Solution: use only one, or if both needed, apply Dropout after BN in FC-only heads.

> **Q: What is the linear scaling rule for learning rate and when does it break down?**
> A: If you multiply batch size by $k$, multiply LR by $k$ (Goyal et al., 2017 "Accurate, Large Minibatch SGD"). Rationale: gradient variance scales as $1/B$, so larger batch = more stable gradient, can take larger step. Breaks at very large batches (>8k): the linear approximation of the loss landscape breaks, curvature matters. Fix: LARS/LAMB optimizers for ultra-large batch training. Also requires warmup — linear scaling doesn't apply in the first epochs before gradients stabilise.

> **Q: A model trains fine on GPU (batch=32) but accuracy drops 5% when switching to batch=1 for CPU inference. What is likely happening?**
> A: Classic BatchNorm variance shift. With BN, `model.eval()` uses running statistics. If the running stats were computed on GPU batches (normalised across 32 samples), they may differ from single-CPU-sample statistics. Fixes: (1) Ensure `model.eval()` is called. (2) Verify running stats are stable (train long enough). (3) Switch to LayerNorm or GroupNorm for single-sample deployment. (4) Fold BN parameters into conv weights post-training (BN fusion) for exact, deterministic inference.


In [ ]:
# ── Module 2 Complete — What We Built ───────────────────────────
print("═══ MODULE 2: TRAINING & OPTIMIZATION — COMPLETE ═══════════════")
print()
notebooks = [
    ("02_training_loss_backprop.ipynb",       "Loss Functions + Backpropagation"),
    ("03_optimizers_lr_scheduling.ipynb",     "Optimizers + LR Scheduling"),
    ("04_regularization.ipynb",               "Regularization"),
    ("05_normalization_initialization.ipynb", "Normalization + Weight Init"),
    ("06_training_loops_and_qna.ipynb",       "Full Training Loops + Q&A"),
]
for nb_file, topic in notebooks:
    print(f"  📓 {nb_file}")
    print(f"     → {topic}")
    print()

print("Topics covered across all 5 notebooks:")
topics = [
    "Loss functions: MSE, MAE, Huber, BCE, CCE",
    "Backpropagation: chain rule, manual worked example, autograd",
    "Vanishing/exploding gradients + gradient clipping",
    "Optimizers: SGD → Momentum → AdaGrad → RMSProp → Adam → AdamW",
    "LR Scheduling: Step, Cosine, Warm Restarts, Warmup",
    "Regularization: L1, L2, Elastic Net, Dropout, Early Stopping",
    "Normalization: BatchNorm, LayerNorm, InstanceNorm, GroupNorm",
    "Weight Init: Zero/Symmetry, Xavier, He (Kaiming), Orthogonal",
    "NumPy full loop: BN + Dropout + Momentum + Early Stopping",
    "PyTorch loop: AMP + clip_grad + scheduler + checkpointing",
]
for t in topics:
    print(f"  ✓ {t}")
